In [1]:
import numpy as np
from sklearnex import patch_sklearn
patch_sklearn()

# Now import your standard ML tools
from sklearn.ensemble import RandomForestClassifier

print("Success! The hardware acceleration engine is active")

Success! The hardware acceleration engine is active


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [2]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

num_rows = 1000000

print("Generating 1,000,000 rows of data... Please wait.")

# 1. Base Demographic Features
age_list = np.random.randint(18, 70, size=num_rows)
smoker_list = np.random.choice([0, 1], size=num_rows, p=[0.82, 0.18])  # ~18% smokers

# 2. Height and Weight (correlated using a rough BMI logic)
height_list = np.round(np.random.normal(168, 10, size=num_rows), 1)
weight = np.round(
    (height_list - 100) * np.random.normal(1.0, 0.15, size=num_rows)
    + np.random.normal(10, 12, size=num_rows),
    1,
)
weight_list = np.clip(weight, 40, 140)

# 3. Categorical Features
cities = ["Mumbai", "Delhi", "Bengaluru", "Hyderabad", "Chennai", "Kolkata", "Pune"]
city_probs = [0.25, 0.22, 0.18, 0.12, 0.10, 0.08, 0.05]
city_list = np.random.choice(cities, size=num_rows, p=city_probs)

occupations = [
    "Software Engineer",
    "Doctor",
    "Teacher",
    "Business Owner",
    "Sales Executive",
    "Banker",
    "Student",
    "Retired",
]
occupation_list = np.random.choice(occupations, size=num_rows)

# 4. Income (LPA) - Influenced by age and occupation
base_income = np.random.exponential(scale=5.0, size=num_rows) + 3.0


def calculate_income(occ, a, inc):
    if occ == "Student":
        return round(np.random.uniform(0, 0.5), 2)
    if occ == "Retired":
        return round(np.random.uniform(3.0, 8.0), 2) if a > 55 else 0.0
    exp_mult = max(1.0, (a - 22) * 0.1)
    if occ == "Software Engineer" or occ == "Doctor":
        return round(inc * 2.5 * exp_mult, 2)
    if occ == "Business Owner":
        return round(inc * 3.0 * np.random.uniform(0.5, 2.0), 2)
    return round(inc * exp_mult, 2)


# Loop using the corrected list names
income_lpa_list = [
    calculate_income(o, a, i)
    for o, a, i in zip(occupation_list, age_list, base_income)
]
income_lpa_list = np.clip(income_lpa_list, 0, 85.0)

# 5. Define occupation weights for target calculation
occupation_weights = {
    "Doctor": 15,
    "Business Owner": 12,
    "Software Engineer": 10,
    "Banker": 8,
    "Sales Executive": 7,
    "Teacher": 5,
    "Retired": 4,
    "Student": 2,
}

# --- Create Single, Unified Dataframe ---
df = pd.DataFrame(
    {
        "age": age_list,
        "weight": weight_list,
        "height": height_list,
        "income_lpa": income_lpa_list,
        "smoker": smoker_list,
        "city": city_list,
        "occupation": occupation_list,
    }
)

# 6. Target Generation Logic (Using vectorized math for speed)
occ_score = df["occupation"].map(occupation_weights).fillna(10).values
age_score = df["age"].values * 0.5
weight_score = (df["weight"].values - 40) * 0.3
smoker_score = df["smoker"].values * 15  # Adding smoker risk to target generation

total_score = (
    occ_score
    + age_score
    + weight_score
    + smoker_score
)

# Vectorized mapping to categories based on percentiles
q25 = np.percentile(total_score, 25)
q50 = np.percentile(total_score, 50)
q75 = np.percentile(total_score, 75)


# Add this print statements to see the actual numbers!
print(f"--- Percentile Thresholds ---")
print(f"q25 (Low Threshold)       : {q25:.2f}")
print(f"q50 (Medium Threshold)    : {q50:.2f}")
print(f"q75 (High Threshold)      : {q75:.2f}")
print(f"-----------------------------")

df["insurance_premium_category"] = np.where(
    total_score < q25,
    "Low",
    np.where(
        total_score < q50,
        "Medium",
        np.where(total_score < q75, "High", "Very High"),
    ),
)

# Map smoker 1/0 to Yes/No for readability
df["smoker"] = df["smoker"].map({1: True, 0: False})

# Save to CSV
output_file = "insurance_dataset_1M.csv"
df.to_csv(output_file, index=False)

print(f"Success! Dataset saved as '{output_file}'")

# --- Ready for ML Pipelines ---
# Define features (X) and target (y)
X = df[["age", "weight", "height", "income_lpa", "smoker", "city", "occupation"]]
y = df["insurance_premium_category"]

print("\nReady for Model Training. Features shape:", X.shape)

Generating 1,000,000 rows of data... Please wait.
--- Percentile Thresholds ---
q25 (Low Threshold)       : 35.37
q50 (Medium Threshold)    : 43.35
q75 (High Threshold)      : 51.53
-----------------------------
Success! Dataset saved as 'insurance_dataset_1M.csv'

Ready for Model Training. Features shape: (1000000, 7)


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [4]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
424636,53,91.4,193.8,0.02,False,Hyderabad,Student,High
153161,58,94.6,183.6,19.85,False,Delhi,Teacher,High
83212,35,71.0,161.9,19.23,False,Delhi,Teacher,Low
873213,30,70.3,147.1,3.62,False,Chennai,Sales Executive,Low
856015,35,98.3,176.0,9.80,False,Mumbai,Doctor,High


In [5]:
df['occupation'].unique()

<StringArray>
[          'Teacher',    'Business Owner',           'Student',
           'Retired',            'Doctor', 'Software Engineer',
   'Sales Executive',            'Banker']
Length: 8, dtype: str

we have to copy the code bcz we r gonna do lots of feature engg on this dataset

In [6]:
df_feat = df.copy()

In [7]:
# Feature 1: BMI
df_feat['bmi'] =df_feat["weight"] /((df_feat["height"]/100)**2)

In [8]:
print(df_feat['bmi'].describe())

count    1000000.000000
mean          27.549480
std            5.532019
min           10.785526
25%           23.729033
50%           27.522855
75%           31.299463
max           56.942420
Name: bmi, dtype: float64


In [9]:
# Feature 2: Age Group
def age_group(age):
  if age < 25:
    return "young"
  elif age <45:
    return "adult"
  elif age < 60:
    return "middle_aged"
  return "senior"

In [10]:
# Feature 2: age_group
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [ ]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 25:
    return "high"
  elif row["smoker"]:
    return "medium"
  else:
    return "low"

In [ ]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [13]:
tier_1_cities = ["Mumbai", "Delhi", "Bengaluru", "Hyderabad", "Chennai", "Bangalore", "Kolkata", "Pune"]
tier_2_cities =[
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varansi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvantapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijaywada", "Tiruchirappali",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareily", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [14]:
# Feature 4: City Tier
def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else: 
    return 3

In [15]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)


In [16]:
df_feat.drop(columns=["age", "weight", "height", "smoker", "city"])[["income_lpa", "occupation", "bmi", "age_group", "lifestyle_risk", "city_tier", "insurance_premium_category"]]

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
0,29.10,Teacher,34.039359,middle_aged,low,1,High
1,63.50,Business Owner,36.841694,senior,low,1,Very High
2,0.46,Student,18.951307,middle_aged,low,1,Low
3,0.00,Retired,28.054746,adult,low,1,Low
4,0.08,Student,31.328310,senior,low,1,High
...,...,...,...,...,...,...,...
999995,17.60,Sales Executive,28.411745,senior,low,1,Very High
999996,26.41,Software Engineer,13.354316,adult,low,1,Low
999997,0.00,Retired,29.794257,adult,low,1,Medium
999998,36.22,Teacher,32.214157,middle_aged,low,1,Very High


In [17]:
# Select Features and Target
X = df_feat[["income_lpa", "occupation", "bmi", "age_group", "lifestyle_risk", "city_tier"]]
y = df_feat["insurance_premium_category"]


In [18]:
X

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier
0,29.10,Teacher,34.039359,middle_aged,low,1
1,63.50,Business Owner,36.841694,senior,low,1
2,0.46,Student,18.951307,middle_aged,low,1
3,0.00,Retired,28.054746,adult,low,1
4,0.08,Student,31.328310,senior,low,1
...,...,...,...,...,...,...
999995,17.60,Sales Executive,28.411745,senior,low,1
999996,26.41,Software Engineer,13.354316,adult,low,1
999997,0.00,Retired,29.794257,adult,low,1
999998,36.22,Teacher,32.214157,middle_aged,low,1


In [19]:
y

0              High
1         Very High
2               Low
3               Low
4              High
            ...    
999995    Very High
999996          Low
999997       Medium
999998    Very High
999999       Medium
Name: insurance_premium_category, Length: 1000000, dtype: str

In [20]:
# Define Categorical and Numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features =["bmi", "income_lpa"]

In [21]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [22]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        # Prevents split on tiny, noisy patterns
        n_estimators=150,         # More trees for a more stable forest
        max_depth= 12  ,         # Limits depth to stop overfitting
        min_samples_split=3,     # Prevents split on tiny, noisy patterns
        class_weight='balanced',  # Fixes the data imbalance from your support column
        random_state=42,          # Keeps results consistent
        n_jobs=-1  
    ))
])

In [23]:
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit the updated model
pipeline.fit(X_train, y_train)


print("Optimized model trained and saved successfully!")

Optimized model trained and saved successfully!


In [24]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
print("Updated Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Updated Accuracy: 0.752855
              precision    recall  f1-score   support

        High       0.66      0.68      0.67     50019
         Low       0.85      0.82      0.84     50059
      Medium       0.65      0.68      0.66     49961
   Very High       0.87      0.83      0.85     49961

    accuracy                           0.75    200000
   macro avg       0.76      0.75      0.75    200000
weighted avg       0.76      0.75      0.75    200000



In [25]:
X_test.sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier
428975,0.35,Student,23.151143,middle_aged,low,1
683149,13.34,Business Owner,25.774258,adult,low,1
415530,35.03,Doctor,30.514335,middle_aged,low,1
247339,8.35,Sales Executive,34.777930,middle_aged,low,1
33380,34.24,Software Engineer,25.386779,young,low,1


In [26]:
print(X_train.columns.tolist())

['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier']


In [ ]:
# Instantly save using joblib
model_filename = 'Insurance_Premium_Predictor_mdl.pkl'
joblib.dump(pipeline, model_filename, compress=3)

print(f"success! Optimized model saved locally as: {model_filename}")

success! Optimized model saved locally as: Insurance_Premium_Predictor_mdl.pkl


: 